# Notebook 2 (Fixed): Data Preprocessing and Cleaning

This version fixes the original runtime bugs in depth:
- `NameError: np is not defined` by importing `numpy as np`.
- `FileNotFoundError: refined_data.csv` by generating it automatically when missing.
- Adds safer parsing for `Measurement Value` without `eval`, with defensive checks for missing columns.


In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
def _safe_parse_measurement_value(v):
    """Safely parse numbers from strings such as '1', '1.5', or '[1.2]'."""
    if pd.isna(v):
        return np.nan

    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)

    s = str(v).strip()
    if not s:
        return np.nan

    # Try direct numeric parse first
    n = pd.to_numeric(s, errors="coerce")
    if pd.notna(n):
        return float(n)

    # Try safe literal parsing for simple containers/scalars
    try:
        lit = ast.literal_eval(s)
        if isinstance(lit, (int, float, np.integer, np.floating)):
            return float(lit)
        if isinstance(lit, (list, tuple)) and len(lit) > 0:
            first = pd.to_numeric(lit[0], errors="coerce")
            return float(first) if pd.notna(first) else np.nan
    except Exception:
        pass

    return np.nan


def prepare_data(df, min_counts=50, one_hot_encode=True):
    """Clean and standardize AMR records for downstream modeling."""
    X = df.copy()

    required_cols = [
        "Measurement", "Measurement Value", "Laboratory Typing Method",
        "Measurement Unit", "Testing Standard", "Measurement Sign",
        "Vendor", "Evidence", "Laboratory Typing Platform",
        "PubMed", "Testing Standard Year", "Resistant Phenotype"
    ]
    missing = [c for c in required_cols if c not in X.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Keep object dtype flexible for string operations
    for col in X.columns:
        if str(X[col].dtype) in {"string", "str"}:
            X[col] = X[col].astype(object)

    m_missing = X["Measurement"].isna() & X["Measurement Value"].notna()
    X.loc[m_missing, "Measurement"] = X.loc[m_missing, "Measurement Value"]

    X.loc[X["Laboratory Typing Method"] == "Disk diffusion", "Laboratory Typing Method"] = "MIC"
    X["Laboratory Typing Method"] = X["Laboratory Typing Method"].fillna("lab_typing_missing")

    # Unit imputation by method
    X.loc[X["Measurement Unit"].isna() & (X["Laboratory Typing Method"] == "Broth dilution"), "Measurement Unit"] = "NTU"
    X.loc[X["Measurement Unit"].isna() & (X["Laboratory Typing Method"] == "Agar dilution"), "Measurement Unit"] = "CFU"
    X.loc[X["Measurement Unit"].isna() & (X["Laboratory Typing Method"] == "MIC"), "Measurement Unit"] = "mg/L"

    X["Testing Standard"] = X["Testing Standard"].replace({
        "eucast": "EUCAST",
        "clsi": "CLSI",
        "EUCAST and CLSI": "CLSI, EUCAST",
    }).fillna("missing")

    X.loc[X["Measurement Sign"].isna() & X["Measurement Value"].notna(), "Measurement Sign"] = "="
    X["Measurement Sign"] = X["Measurement Sign"].fillna("no_sign")

    X["Measurement Value"] = X["Measurement Value"].apply(_safe_parse_measurement_value)

    X = X.fillna({
        "Vendor": "vendor_missing",
        "Evidence": "Laboratory Method",
        "Laboratory Typing Method": "typing_method_missing",
        "Laboratory Typing Platform": "typing_platform_missing",
        "Measurement Value": 0.0,
        "Measurement Unit": "unit_missing",
    })

    # Drop sparse columns robustly (fixes np NameError and keeps logic explicit)
    non_null_counts = X.notna().sum()
    empty_cols = non_null_counts[non_null_counts < min_counts].index.tolist()
    X = X.drop(columns=empty_cols, errors="ignore")

    X = X.drop(columns=["Measurement", "Evidence"], errors="ignore")

    if one_hot_encode:
        X["PubMed"] = X["PubMed"].notna().astype(int)
        X["Testing Standard Year"] = X["Testing Standard Year"].notna().astype(int)
    else:
        X["PubMed"] = X["PubMed"].fillna("missing_pubmed")
        X["Testing Standard Year"] = X["Testing Standard Year"].fillna(0)

    return X[X["Resistant Phenotype"].notna()].reset_index(drop=True)


In [3]:
data_path = Path('../data/raw/data.csv')
if not data_path.exists():
    raise FileNotFoundError(f"Missing input file: {data_path.resolve()}")

data = pd.read_csv(data_path, encoding='latin1')
refined_data = prepare_data(data, min_counts=50, one_hot_encode=True)
refined_data.to_csv('../data/processed/refined_data.csv', index=False)

print('data/processed/refined_data.csv created')
print('shape:', refined_data.shape)
refined_data.head()


/var/folders/80/wn8x154d6fn2js7st96hq8p00000gn/T/ipykernel_7761/937369315.py:5: DtypeWarning: Columns (0: Source) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path, encoding='latin1')


refined_data.csv created
shape: (95178, 16)


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,Laboratory Typing Method Version,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Source,PubMed
0,562,562.144245,Escherichia coli ERR7221502,meropenem,Susceptible,no_sign,0.0,NTU,Broth dilution,NaN,VITEK 2,BioMerieux,EUCAST,1,NaN,1
1,562,562.100003,Escherichia coli c1b4da3e-7bb9-11e9-a8d3-68b59...,trimethoprim/sulfamethoxazole,Susceptible,no_sign,0.0,mg/L,MIC,NaN,typing_platform_missing,vendor_missing,EUCAST,0,NaN,0
2,562,562.658140,Escherichia coli strain V139,cefepime,Resistant,no_sign,0.0,unit_missing,lab_typing_missing,NaN,typing_platform_missing,vendor_missing,missing,0,NaN,1
3,562,562.770300,Escherichia coli strain 503320,ceftriaxone,Susceptible,<=,1.0,mg/L,Broth dilution,NaN,VITEK 2,BioMerieux,EUCAST,0,NaN,1
4,562,562.230900,Escherichia coli O22:H1 strain ECO0731,amoxicillin,Susceptible,no_sign,0.0,CFU,Agar dilution,NaN,typing_platform_missing,vendor_missing,missing,0,NaN,1


In [4]:
refined_path = Path('../data/processed/refined_data.csv')
if not refined_path.exists():
    print('refined_data.csv not found; generating from ../data/raw/data.csv...')
    source = pd.read_csv('../data/raw/data.csv', encoding='latin1')
    refined_data = prepare_data(source, min_counts=50, one_hot_encode=True)
    refined_data.to_csv(refined_path, index=False)

refined_data = pd.read_csv(refined_path)
refined_data.info()
refined_data.head()


<class 'pandas.DataFrame'>
RangeIndex: 95178 entries, 0 to 95177
Data columns (total 16 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Taxon ID                          95178 non-null  int64  
 1   Genome ID                         95178 non-null  float64
 2   Genome Name                       95178 non-null  str    
 3   Antibiotic                        95178 non-null  str    
 4   Resistant Phenotype               95178 non-null  str    
 5   Measurement Sign                  95178 non-null  str    
 6   Measurement Value                 95178 non-null  float64
 7   Measurement Unit                  95178 non-null  str    
 8   Laboratory Typing Method          95178 non-null  str    
 9   Laboratory Typing Method Version  161 non-null    str    
 10  Laboratory Typing Platform        95178 non-null  str    
 11  Vendor                            95178 non-null  str    
 12  Testing Standar

,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,Laboratory Typing Method Version,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Source,PubMed
0,562,562.144245,Escherichia coli ERR7221502,meropenem,Susceptible,no_sign,0.0,NTU,Broth dilution,NaN,VITEK 2,BioMerieux,EUCAST,1,NaN,1
1,562,562.100003,Escherichia coli c1b4da3e-7bb9-11e9-a8d3-68b59...,trimethoprim/sulfamethoxazole,Susceptible,no_sign,0.0,mg/L,MIC,NaN,typing_platform_missing,vendor_missing,EUCAST,0,NaN,0
2,562,562.658140,Escherichia coli strain V139,cefepime,Resistant,no_sign,0.0,unit_missing,lab_typing_missing,NaN,typing_platform_missing,vendor_missing,missing,0,NaN,1
3,562,562.770300,Escherichia coli strain 503320,ceftriaxone,Susceptible,<=,1.0,mg/L,Broth dilution,NaN,VITEK 2,BioMerieux,EUCAST,0,NaN,1
4,562,562.230900,Escherichia coli O22:H1 strain ECO0731,amoxicillin,Susceptible,no_sign,0.0,CFU,Agar dilution,NaN,typing_platform_missing,vendor_missing,missing,0,NaN,1
